# Diacritics Restoration project
author: [Jakub Łabuz](https://github.com/jakseluz)

## Introduction

The project focuses on diacritics restoration in Polish language words taking the context into account.

e.g. Labuz -> Łabuz


### Research
Articles which I found to be adequate for the problem:
- „Diacritics Restoration Using Neural Networks”\
(Jakub N´aplava, Milan Straka, Pavel Straˇn´ak, Jan Hajiˇc, 2018)
- [„Diacritics Restoration using BERT with Analysis on Czech language”\
(Jakub N´aplava, Milan Straka, Jana Strakov´a, 2021)](https://arxiv.org/abs/2105.11408)
- [„Correcting Diacritics and Typos with a ByT5 Transformer Model”\
(Lukas Stankeviˇcius, Mantas Lukoˇseviˇcius, Jurgita Kapoˇci¯ut˙e-Dzikien˙e,
Monika Briedien˙e, Tomas Krilaviˇcius, 2022)](https://arxiv.org/abs/2201.13242)
- [„Dilated Convolutional Neural Networks for Lightweight Diacritics
Restoration”\
(B´alint Csan´ady, Andr´as Luk´acs, 2022)](https://arxiv.org/abs/2201.06757)
- [„Romanian Diacritics Restoration Using Recurrent Neural Networks”\
(Stefan Ruseti, Teodor-Mihai Cotet, and Mihai Dascalu, 2020)](https://arxiv.org/abs/2009.02743).


### Main possible approaches
- character-level classification
- transformers connected with an external LLM
- sequence-to-sequence.


### Project assumptions
- self-supervised learning
- batch generating during the learning process - by diacritics removal.


### Dataset I used
- Polish Wikipedia, using [datasets library](https://huggingface.co/docs/datasets/index) - large and fully sufficient for learning.
Wikimedia Wikipedia (PL):
[https://huggingface.co/datasets/wikimedia/wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia):
    ```python
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", "20231101.pl")
    ```


### Other datasets - promising but not needed here:
- CulturaX (Polish subset):
https://huggingface.co/datasets/uonlp/CulturaX
- hand-annotated million NJKP corpus:
https://nkjp.pl/index.php?page=14lang=0
- CLARIN-PL corpuses:
https://clarin-pl.eu/catalog/resources - e.g. Parliamentary sessions of Sejm & Senat RP (300 milion of
tokens)
- PolEval (NLP competitions):
http://poleval.pl/ - e.g. to compare used data with competitors solutions.


### Metrics for evaluation
- ~~accuracy~~ - not especially helpful - can be good even when a model does not work (diacritics percentages in words are quite low)
- WER (Word Error Rate) - mistaken word percentage 
- CER (Character Error Rate) - mistaken character percentage
- DER (Diacritic Error Rate) - mistaken diacritics percentage.

From the above, I have chosen CER to be the most valuable indicator.
That is beacuse models can - not only restore diacritics where they are expected to do it - but also where the letter should be untouched.
CER takes into account both situations and present the general model efficiency when considering the project topic.

## Project code

In [1]:
%load_ext autoreload
%autoreload 2

### Import Pytorch and print the configuration information

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.12.0+cu130
CUDA available: True
CUDA version: 13.0
GPU name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


#### Dataset 'preconfiguration tests' - if you want to check how the dataset looks like

In [3]:
from diacritics_restoration.utils import get_wikipedia_data

lista = [
    text for text in get_wikipedia_data(num_articles=5).head()["text"].tolist()
]
print("Original texts:")
print(lista)

Polish Wikipedia successfully loaded!
Wikipedia dataset converted to DataFrame!
Original texts:
['HMS „Lancaster” – nazwa noszona przez siedem okrętów brytyjskiej Royal Navy, pochodząca od miasta Lancaster:\n  – 80-działowy okręt liniowy drugiej rangi (second rate) zwodowany w 1694, przebudowany w 1722, rozebrany w 1743.\n  – 66-działowy okręt liniowy trzeciej rangi (third rate) zwodowany w 1749, rozebrany w 1773.\n  – 64-działowy okręt liniowy trzeciej rangi (third rate), pierwotnie zaprojektowany jako statek handlowy typu East Indiaman, zwodowany w 1797, rozebrany w 1832.\n  – 58-działowy okręt liniowy czwartej rangi (fourth rate) zwodowany w 1823, sprzedany w 1864.\n  – krążownik pancerny typu Monmouth zwodowany w 1902, sprzedany w 1920.\n HMS „Lancaster” – amerykański niszczyciel typu Wickes (ex-USS „Philip”) przekazany Royal Navy w 1940, złomowany w 1947.\n  – fregata rakietowa typu 23 (Duke) zwodowana w 1990, w czynnej służbie.\n\nPrzypisy \n\nLancaster', 'Kosmos 96 () – radzieck

In [3]:
from diacritics_restoration.utils import (
    get_wikipedia_data,
    DiacriticsDataset,
)

import torch
from torch.utils.data import DataLoader
from torch import nn

#### data preparation

In [4]:
def get_dataset_and_dataloader(
    num_articles=10_000, batch_size=256, shuffle=False
) -> tuple[DiacriticsDataset, DataLoader]:
    dataset = DiacriticsDataset(
        get_wikipedia_data(num_articles=num_articles)["text"].tolist()
    )
    print("Dataset size:", len(dataset))
    return dataset, DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [6]:
dataset, dataloader = get_dataset_and_dataloader(
    num_articles=10000, batch_size=512, shuffle=True
)

Polish Wikipedia successfully loaded!
Wikipedia dataset converted to DataFrame!
Dataset size: 145134


### Dilated 1D CNN - first

In [7]:
from diacritics_restoration.models import DiacriticsCNN
from diacritics_restoration.utils import train_CNN_model

#### model definition

In [8]:
model = DiacriticsCNN(vocab_size=dataset.processor.vocab_size)
criterion = nn.CrossEntropyLoss(ignore_index=dataset.processor.pad_token_id)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
from torchinfo import summary

summary(model, input_size=(256, 256), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
DiacriticsCNN                            [256, 105, 256]           --
├─Embedding: 1-1                         [256, 256, 128]           13,440
├─Conv1d: 1-2                            [256, 256, 256]           98,560
├─ModuleList: 1-3                        --                        --
│    └─ResidualDilatedBlock: 2-1         [256, 256, 256]           --
│    │    └─Conv1d: 3-1                  [256, 256, 256]           196,864
│    │    └─BatchNorm1d: 3-2             [256, 256, 256]           512
│    │    └─Conv1d: 3-3                  [256, 256, 256]           196,864
│    │    └─BatchNorm1d: 3-4             [256, 256, 256]           512
│    └─ResidualDilatedBlock: 2-2         [256, 256, 256]           --
│    │    └─Conv1d: 3-5                  [256, 256, 256]           196,864
│    │    └─BatchNorm1d: 3-6             [256, 256, 256]           512
│    │    └─Conv1d: 3-7                  [256, 256, 256]   

#### training

In [11]:
print("Starting training...")
train_CNN_model(
    model, dataloader, epochs=40, criterion=criterion, optimizer=optimizer
)

Starting training...


Epoch 1/40: 100%|██████████| 284/284 [02:20<00:00,  2.02it/s, loss=0.0117]


Epoch 1/40 - Average Loss: 0.0766
Best model saved (models/DiacriticsCNN/DiacriticsCNN_2026-05-26_18-49-50_best_model.pt) with loss: 0.07659040550491444


Epoch 2/40: 100%|██████████| 284/284 [02:20<00:00,  2.02it/s, loss=0.00762]


Epoch 2/40 - Average Loss: 0.0094
Best model saved (models/DiacriticsCNN/DiacriticsCNN_2026-05-26_18-49-50_best_model.pt) with loss: 0.00939667247839167


...

Epoch 5/40 - Average Loss: 0.0037
Best model saved (models/DiacriticsCNN/DiacriticsCNN_2026-05-26_18-49-50_best_model.pt) with loss: 0.0036568812383207636


Epoch 6/40: 100%|██████████| 284/284 [02:21<00:00,  2.01it/s, loss=0.003]  


Epoch 6/40 - Average Loss: 0.0030
Best model saved (models/DiacriticsCNN/DiacriticsCNN_2026-05-26_18-49-50_best_model.pt) with loss: 0.002964545552894263


Epoch 7/40:  32%|███▏      | 92/284 [00:46<01:36,  1.99it/s, loss=0.00253]


KeyboardInterrupt: 

#### model evaluation

In [5]:
from diacritics_restoration.utils import DiacriticsRestorer
from diacritics_restoration.models import DiacriticsCNN

import torch

##### the latest restorer

In [6]:
import glob
import os
from diacritics_restoration.utils.processor import CharacterProcessor


def load_latest_restorer(path: str) -> DiacriticsRestorer:
    print("Loading best model weights...")
    model_files = glob.glob(path)
    latest_model_file = max(model_files, key=os.path.getctime)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = CharacterProcessor()
    model = DiacriticsCNN(vocab_size=processor.vocab_size)
    state = torch.load(latest_model_file, map_location=device)
    model.load_state_dict(state)
    model.to(device=device)
    model.eval()
    restorer = DiacriticsRestorer(
        model=model, processor=processor, device=device
    )
    return restorer

##### sample uses

In [7]:
def test(
    restorer: DiacriticsRestorer,
    test_text: str = "To jest przykladowy tekst bez znakow diakrytycznych.",
) -> None:
    restored_text = restorer.restore(test_text)
    print("Original:", test_text)
    print("Restored:", restored_text)


restorer = load_latest_restorer(path="models/DiacriticsCNN/*best_model.pt")

test(restorer)
test(restorer, "Zazolc to gory, a na niej siedzi zielony zolw.")
test(restorer, "Wczoraj bylem w sklepie i kupilem mleko oraz chleb.")
test(restorer, "Czy moglbys mi powiedziec, gdzie jest najblizsza stacja metra?")

Loading best model weights...
Original: To jest przykladowy tekst bez znakow diakrytycznych.
Restored: To jest przykładowy tekst bez znaków diakrytycznych.
Original: Zazolc to gory, a na niej siedzi zielony zolw.
Restored: Zażolć to góry, a na niej siedzi zielony żółw.
Original: Wczoraj bylem w sklepie i kupilem mleko oraz chleb.
Restored: Wczoraj byłem w sklepie i kupiłem mleko oraz chleb.
Original: Czy moglbys mi powiedziec, gdzie jest najblizsza stacja metra?
Restored: Czy mógłbyś mi powiedzieć, gdzie jest najbliższą stacją metra?


##### **CER** - Character Error Rate (Dilated 1D CNN)

In [8]:
print(restorer.calculate_history_character_error_rate())

0.08056872037914692


##### Evaluate on random articles

In [9]:
from diacritics_restoration.utils.testing import evaluate_restorer_on_articles

evaluate_restorer_on_articles(
    restorer=restorer, num_articles=1000, batch_size=256
)

Polish Wikipedia successfully loaded!
Wikipedia dataset converted to DataFrame!
Dataset size: 13974

Example 0
PRED: Parafia pw. Ducha Świętego w Płocku <UNK> rzymskokatolicka parafia należąca do dekanatu płockiego zachodniego, diecezji płockiej, metropolii warszawskiej. Parafia powstała <UNK><UNK> marca <UNK><UNK><UNK><UNK> roku. Jej obecnym proboszczem jest ks. Zbigniew Jerzy Zbrzeżny.<UNK><UNK>Miejsca s
TRUE: Parafia pw. Ducha Świętego w Płocku <UNK> rzymskokatolicka parafia należąca do dekanatu płockiego zachodniego, diecezji płockiej, metropolii warszawskiej. Parafia powstała <UNK><UNK> marca <UNK><UNK><UNK><UNK> roku. Jej obecnym proboszczem jest ks. Zbigniew Jerzy Zbrzezny.<UNK><UNK>Miejsca ś
Differences: 2 / 292

Example 1
PRED: niego, diecezji płockiej, metropolii warszawskiej. Parafia powstała <UNK><UNK> marca <UNK><UNK><UNK><UNK> roku. Jej obecnym proboszczem jest ks. Zbigniew Jerzy Zbrzeżny.<UNK><UNK>Miejsca święte<UNK><UNK>Kościół parafialny <UNK>Kościół parafialny pw. Dob

### ByT5

In [2]:
# %pip install -U transformers datasets accelerate torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")
